---
title: "Change Planner Agent: Architecture and Contracts"
draft: true
description: "Define the repository search, evidence, regression-analysis, memory, workflow, and evaluation contracts for a Change Planner Agent built with LangGraph."
categories: [agents, langgraph, ai-engineering, search, retrieval, memory, software-engineering, reproducibility]
---

This chapter is the course-wide map for a workflow that turns a bounded change request into a reviewed, evidence-backed change plan. The agent investigates a versioned repository snapshot, relates code and configuration to tests and history, forms regression hypotheses, performs permitted verification, and reports both conclusions and unresolved questions.

The project is deliberately introspective: its primary search space is the system's own artifacts and the agent's prior investigations, not the open web. LangGraph is useful when evidence gathering needs named state transitions, dynamic fan-out, bounded refinement, durable checkpoints, revision-aware memory, or human interrupts. It does not parse code, make retrieval relevant, prove that a test covers a behavior, observe an absent production environment, or decide that a rollout is safe.


## The running project and its thesis

The backing project is the **Change Planner Agent with LangGraph**. Its canonical scenario asks for a dry-run mode on a state-changing CLI command in a pinned Python repository. The request appears small, but a defensible plan must recover the command entry point, side-effect boundary, public interface, configuration assumptions, existing tests, documentation, and prior changes; identify behavior that must remain stable; and expose any path that lacks verification.

Additional fixture scenarios cover cross-module API changes, configuration migrations, altered retry behavior, test gaps, and regressions introduced by apparently local edits. Each scenario is tied to a repository revision and has curated evidence, affected-surface, related-test, and regression expectations.

The project thesis is:

> A change investigation should earn graph structure only after search and code-analysis baselines are measurable, and only where evidence sufficiency, parallel inquiry, bounded verification, persistence, memory, or review changes control flow.

Chapter 01 tests that thesis against direct retrieval and a staged Python planner. Later chapters add LangGraph primitives only when a concrete investigation contract needs them.


## Scope and ownership

The course owns the end-to-end investigation contract, pinned repositories, indexes, evidence schemas, deterministic search and analysis tools, model adapters, graph topology, fault plans, memory policy, evaluation cases, and exported artifacts. LangGraph supplies runtime primitives for composing and executing the stateful control flow.

| Concern | Course and backing project | LangGraph runtime |
|---|---|---|
| Repository evidence | Snapshot identity, parsers, chunks, symbols, tests, diffs, source locations, and fingerprints | No parser, source-of-truth policy, or automatic understanding of behavior |
| Search and ranking | Literal search, BM25, embeddings, fusion, reranking, graph traversal, freshness checks, and relevance metrics | Nodes can select and repeat strategies, but the runtime does not make retrieval relevant |
| Relationships and verification | Candidate impact edges, test links, regression hypotheses, allowed commands, observed results, and citation checks | State and routes expose what was attempted and what remains unresolved |
| Control flow | Search budgets, branch contracts, sufficiency policy, review semantics, export rules, and terminal reasons | `StateGraph`, reducers, conditional edges, `Send`, `Command`, and `interrupt` |
| Durability and memory | Repository namespaces, admission rules, provenance, invalidation, idempotent effects, and migration policy | Checkpointers, threads, stores, replay, forks, and subgraph persistence |
| Evaluation | Gold evidence, seeded regressions, retrieval metrics, impact metrics, memory cases, and ablations | No correctness oracle for code behavior or production safety |

The course is Python-first and repository-bounded. It does not build a general coding agent, edit source files, merge changes, deploy systems, inspect an unavailable production environment, or promise that static evidence captures every runtime dependency. Rollout, rollback, and observability sections are planning artifacts whose claims must remain tied to supplied evidence and labeled assumptions.


## Target workflow and artifact lineage

The graph coordinates an investigation around deterministic search and analysis services. It is not a container for every parser, retriever, scorer, or validator.

```{mermaid}
flowchart TD
    request["ChangeRequest + execution policy"] --> intake["validate scope and RepositorySnapshot"]
    memory["revision-aware memory store"] --> recall["retrieve and revalidate prior investigations"]
    intake --> plan["plan evidence requirements"]
    recall --> plan
    plan --> dispatch["Send investigation tasks"]
    dispatch --> code["code and symbol search"]
    dispatch --> tests["test and contract search"]
    dispatch --> config["configuration and documentation search"]
    dispatch --> history["Git history and diff search"]
    code --> join["validate branches and fuse evidence"]
    tests --> join
    config --> join
    history --> join
    join --> impact["build impact model and regression hypotheses"]
    impact --> sufficient{"evidence sufficient?"}
    sufficient -->|no: searchable gap| plan
    sufficient -->|no: verification needed| verify["run explicitly allowed targeted checks"]
    verify --> impact
    sufficient -->|yes| draft["draft versioned change plan"]
    draft --> review{"interrupt: human review"}
    review -->|request evidence| plan
    review -->|edit or reject| draft
    review -->|approve| audit["verify citations, freshness, status, and memory candidates"]
    audit --> export["export Markdown plan + JSON investigation record"]
    audit --> memory
    checkpoint[("checkpoint / investigation thread")] -. persists .-> review
```

The artifact lineage is:

1. A validated `ChangeRequest` identifies the requested outcome, repository, target revision, allowed scope, constraints, and execution policy. A `RepositorySnapshot` fixes the source and index identities used by the run.
2. Planning produces bounded `InvestigationTask` objects for current behavior, affected surfaces, tests, configuration and documentation, and relevant history.
3. Each branch returns versioned `Evidence` objects with source kind, path or commit, symbol when available, recoverable location, content fingerprint, retrieval trace, and text.
4. Fusion produces candidate relationships, `RegressionHypothesis` objects, and `TestLink` objects. These records distinguish retrieved proximity, engineering inference, and observed verification.
5. The sufficiency policy either accepts the evidence, schedules a bounded search refinement, requests an allowed test, or terminates with an explicit gap.
6. Drafting produces a versioned `ChangePlanArtifact`. Review may approve it, edit plan fields, reject it, or request more evidence. Final audit verifies citations and decides which reviewed findings may become memory candidates.
7. Export writes the human-readable plan and machine-readable investigation record. The run record preserves routes, searches, scores, tests, review actions, costs, terminal status, and memory effects.

A course result is this lineage plus the exported plan, not a fluent implementation proposal detached from the repository revision and the evidence that supports it.


## State is the investigation contract

The cumulative graph uses typed state and reducers only where parallel branches legitimately contribute values:

```python
class ChangeState(TypedDict, total=False):
    request: dict[str, Any]
    snapshot: dict[str, Any]
    tasks: list[dict[str, Any]]
    branch_results: Annotated[list[dict[str, Any]], operator.add]
    evidence: list[dict[str, Any]]
    relationships: list[dict[str, Any]]
    hypotheses: list[dict[str, Any]]
    test_links: list[dict[str, Any]]
    memory_hits: list[dict[str, Any]]
    artifact: dict[str, Any]
    review: dict[str, Any]
    search_rounds: int
    verification_rounds: int
    status: str
    terminal_reason: str
    events: Annotated[list[str], operator.add]
```

The list reducer is not a completeness check. It lets independent investigation branches contribute results, while the join verifies expected task IDs, duplicate absence, snapshot identity, completion status, and the policy for gaps. Scalar fields such as round counts and status remain single-owner updates.

Runtime context carries retrievers, parsers, repository access, the model adapter, fault plan, test runner, and effect ledger because those injected dependencies are not durable facts about the change request. Persisted state contains serializable identities and results, not open repository handles or model clients. Chapter 02 makes these ownership rules executable.


## Evidence preserves what search found

The source contract starts with the exact repository snapshot and narrows each result to a recoverable location:

```python
Evidence(
    id="change-cli@8f2c1d:src/change_cli/commands.py:88-121",
    repository="fixture/change-cli",
    revision="8f2c1d",
    source_kind="code",
    path="src/change_cli/commands.py",
    symbol="clear_outputs",
    start_line=88,
    end_line=121,
    content_hash="sha256:...",
    retrieval=[
        {"method": "bm25", "rank": 2, "score": 7.31},
        {"method": "dense", "rank": 4, "score": 0.78},
    ],
    text="...",
)
```

Literal and BM25 search recover exact names and terminology. Dense retrieval can recover semantically related passages that use different words. Symbol and dependency traversal supply structural candidates, while Git and test retrieval add temporal and behavioral evidence. Hybrid retrieval combines ranked lists before a reranker scores the smaller candidate set.

One fusion baseline is reciprocal rank fusion. For document $d$, retriever set $R$, rank $r_j(d)$, and smoothing constant $k$:

$$
\operatorname{RRF}(d) = \sum_{j \in R} \frac{1}{k + r_j(d)}.
$$

The score rewards evidence retrieved by multiple methods without pretending their raw scores share a scale. The course still measures each component separately because fusion can hide a weak retriever behind a stronger one.

A nearby symbol, co-changed file, importing test, or high similarity score is a candidate relationship, not proof of impact. `TestLink` and `RegressionHypothesis` records preserve the evidence supporting and refuting a claim, the checks that could resolve it, and whether verification was observed or merely proposed. Chapter 03 builds these contracts; Chapter 08 tests whether they improve retrieval and regression localization.


## Checkpoints and memory solve different problems

A checkpoint answers, “How can this investigation resume from a known graph state?” Long-term memory answers, “What validated experience from earlier investigations might help with this repository and revision?” Treating the two as synonyms produces resumable workflows that remember nothing useful, or searchable memories with no reliable execution history.

The course separates four scopes:

| Scope | Contents | Lifetime |
|---|---|---|
| Working state | Current tasks, evidence, hypotheses, budgets, review, and status | One investigation thread |
| Episodic memory | Prior change requests, searches attempted, tests run, outcomes, and reviewer feedback | Across investigations, with repository and revision provenance |
| Semantic repository memory | Reviewed architecture facts, terminology, invariants, ownership boundaries, and known test relationships | Until its supporting evidence changes or review invalidates it |
| Procedural memory | Search strategies and investigation patterns that helped on this repository class | Across compatible repositories and tool versions |

A memory record carries its repository namespace, source revision, supporting evidence IDs and fingerprints, creation investigation, confidence, review status, and invalidation metadata. Retrieval first finds potentially useful memory; a freshness gate then checks whether the target revision changed its supporting sources. Stale memory can guide a new search, but it cannot count as current evidence until revalidated.

The admission policy is as important as retrieval. The agent should not store every model sentence or transient observation. Reviewed findings, observed test outcomes, resolved aliases, and successful search procedures are candidates; unsupported inferences and secrets are rejected. Chapter 07 implements admission, consolidation, retrieval, update, invalidation, and deletion, then compares valid, irrelevant, conflicting, and stale-memory cases.


## Execution paths and resource expectations

The execution profiles vary repository size, index construction, model adapter, and durability boundary without changing the evidence schemas or evaluation identities.

| Path | Resources | What remains fixed | What can vary |
|---|---|---|---|
| Deterministic smoke | Python, pinned fixture repositories, committed compact index artifacts, scripted planner, in-memory or SQLite saver | Schemas, scenario answers, graph routes, fault plans, change-plan shape, and process checks | Saver and selected workflow variant |
| Standard local search | Local index rebuild with pinned lexical, embedding, and optional reranking profiles | Repository revisions, relevance judgments, metric definitions, and artifact interfaces | CPU or accelerated inference, batch size, and measured latency |
| Real repository integration | Learner-selected local repository and explicit read/test permissions | Evidence provenance, graph contracts, memory freshness, and export validation | Languages, repository conventions, available tests, and answer completeness |
| Optional live adapter | Explicit model API credential and network access | Typed request, deterministic tools, graph state, evidence checks, and output boundaries | Query decomposition, synthesis, latency, cost, and model failure surface |

No paid service is required for the canonical path. The deterministic smoke path keeps failures reproducible; the standard path supplies the meaningful search comparison. A real repository run may reveal unsupported languages, generated code, dynamic dependencies, unavailable services, or missing tests, and the honest result may be an incomplete plan.

For independent investigation branches, parallelism changes the critical path:

$$
T_{\text{parallel}} \approx \max_i T_i,
\qquad
T_{\text{sequential}} = \sum_i T_i.
$$

The approximation is useful only when branches are independent and the join accounts for every scheduled task. Parallel search does not reduce the evidence requirements or excuse a missing test or history branch.


## Evaluation standard

The course evaluates search, change analysis, memory, and workflow behavior separately so one strong layer cannot conceal another weak one.

| Evaluation tier | What it establishes | What it cannot establish |
|---|---|---|
| Retrieval benchmark | Whether a retriever recovers curated files, symbols, tests, configuration, documentation, and commits for frozen questions | That retrieved evidence proves behavioral impact |
| Seeded change scenarios | Whether the agent identifies expected affected surfaces, related tests, regression hypotheses, and unresolved gaps on pinned commits | That the fixtures represent every production architecture |
| Memory cases | Whether valid prior investigations help, and whether irrelevant, conflicting, or stale memories are ignored or revalidated | That a memory policy is safe for every repository or organization |
| Process and ablations | Whether routes terminate, branches join accountably, reviews alter state, effects remain idempotent, and complexity earns measurable value | That a procedurally sound workflow produced a safe production plan |
| Real repository integration | Whether the interfaces survive realistic repository conventions and honest missing evidence | A frozen correctness score when no complete answer key exists |

Representative metrics remain disaggregated:

| Layer | Metrics or checks |
|---|---|
| Retrieval | Recall at k, reciprocal rank, nDCG, evidence precision, index freshness, latency, and cost |
| Impact analysis | Affected-file and affected-symbol recall, relationship precision, and unsupported-impact rate |
| Tests and regressions | Related-test recall, seeded-regression localization rank, verification coverage, and false reassurance rate |
| Memory | Helpful-memory rate, stale-memory rejection, contradiction handling, unsupported-memory use, and repeated-investigation cost |
| Workflow | Valid transitions, termination, branch accounting, retry ownership, review effects, restart behavior, and duplicate-effect count |
| Artifact | Citation completeness, revision consistency, explicit unknowns, test-plan coverage, and separation of observed facts from proposals |

The ablation ladder compares literal search, BM25, dense retrieval, hybrid fusion, reranking, structural traversal, agentic refinement, and memory augmentation on fixed scenarios. The graph variant must also be compared with a direct pipeline and staged Python planner. Chapter 08 reports gains, regressions, latency, and cost rather than collapsing them into one “agent quality” score.


## Chapter-by-chapter build

| Chapter | New mechanism | Artifact or evidence contributed |
|---:|---|---|
| 01 | Task-boundary rubric and framework-free baselines | Direct search and staged planning results that establish where graph structure might earn its cost |
| 02 | Typed `StateGraph`, conditional edges, reducers, and `Command` | A minimal investigation graph with explicit state ownership and bounded evidence refinement |
| 03 | Repository ingestion, BM25, dense retrieval, fusion, reranking, symbols, tests, and Git evidence | Versioned evidence records and retrieval reports tied to frozen relevance judgments |
| 04 | Typed routes, retry policy, budgets, and terminal reasons | Fault matrix for ambiguity, stale indexes, missing evidence, tool faults, and exhausted verification |
| 05 | `Send` fan-out, accountable joins, and impact synthesis | Parallel code, test, configuration, documentation, and history investigation with branch-completeness checks |
| 06 | `interrupt` and `Command(resume=...)` | Human review protocol whose approval, edits, rejection, and evidence requests change the plan or graph route |
| 07 | SQLite checkpointers, replay, forks, stores, and memory policies | Restart evidence plus episodic, semantic, and procedural memory with revision-aware invalidation |
| 08 | Retrieval metrics, seeded regressions, memory cases, process checks, and ablations | Layered scorecard comparing search, analysis, graph, and memory variants |
| 09 | Integrated graph, CLI, and export | Final Markdown change plan, JSON investigation record, challenge results, failure runs, and decision memo |

The chapters preserve one artifact lineage. Reusable computation accumulates in the backing project, while notebooks introduce a contract, run the shared implementation, and interpret stored evidence. Later chapters refer to earlier identities and reports instead of rebuilding parallel versions of the system.


## Acceptance conditions and non-goals

The final course claim is accepted only when:

- direct retrieval, a staged Python planner, and the graph workflow are compared on the same frozen scenarios;
- every material claim in a change plan resolves to revision-matched evidence or is labeled as an assumption, proposal, or unresolved question;
- lexical, dense, hybrid, reranked, and structural retrieval variants report fixed relevance metrics, latency, and cost;
- fan-out and fan-in account for every scheduled investigation task exactly once;
- code, test, configuration, documentation, and history relationships distinguish candidates, inferences, and observed verification;
- transient failures retry only where repetition can change the outcome, while stale indexes, unavailable evidence, disallowed tests, and exhausted budgets terminate through named routes;
- review decisions change graph state or the versioned plan artifact;
- a SQLite process restart resumes the same investigation without duplicate search, test, memory, or export effects;
- valid memory can help a repeated investigation while stale or conflicting memory is rejected, downgraded, or revalidated;
- seeded regressions are evaluated separately from retrieval relevance and workflow correctness; and
- the capstone publishes both the human-readable change plan and machine-readable record, including failed and incomplete cases.

The course does not promise full language coverage, whole-program static analysis, automatic proof of test adequacy, access to unsupplied production state, autonomous code changes, deployment authorization, or a universal rule that change planning should use LangGraph. Its transferable result is a measurable search and memory system wrapped in explicit orchestration, with enough provenance to review how a plan was formed.

## Reading the numbered chapters

Read the chapters in order. Chapter 01 establishes the graph boundary; Chapter 02 gives the investigation typed state; Chapters 03–05 build search, recovery, and impact analysis; Chapters 06–08 add review, durability, memory, and evaluation; Chapter 09 assembles the Change Planner.

The central handoff is from retrieved proximity to justified change impact. A high-scoring code result can still be irrelevant, an importing test can still miss the changed behavior, and a remembered architecture fact can become stale after one commit. The course treats search, relationships, verification, memory, and graph control as separate contracts that must agree before a plan can be trusted.

:::{.callout-warning}
The capstone produces decision support, not production authorization. Validate the actual environment, consult the responsible engineers, and follow the organization's review, security, change-management, and rollback procedures before acting on a generated plan.
:::
